# 0 · Setup & smoke test

This notebook installs **langchain-engram** from source and verifies your
Engram credentials with a tiny add → wait → search round trip.

You need an Engram API key — create one at
<https://console.weaviate.cloud/engram>. No LLM key is required here.

## Setup — pick **one** option

**Option A — editable install (recommended).** Installs this package *and* its
dependencies from source; edits to `langchain_engram/` are picked up after a
kernel restart.

**Option B — import from local source (no install of `langchain-engram`).** Adds
the repo root to `sys.path` so `import langchain_engram` resolves straight to the
source tree. You still need the runtime dependencies available — the easiest is to
launch Jupyter from the repo's environment: `uv run --with jupyter jupyter lab`.

In [ ]:
# Option A — editable install from source.
%pip install -q -e ".."

In [ ]:
# Option B — use langchain-engram straight from the local source tree (no install).
# Run this INSTEAD of Option A.
import sys
from pathlib import Path

repo_root = Path.cwd().parent  # notebooks/ -> repo root (holds langchain_engram/)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import langchain_engram

print('langchain_engram imported from:', langchain_engram.__file__)

Provide your Engram API key (stored only in this kernel's environment):

In [ ]:
import getpass
import os

if not os.environ.get("ENGRAM_API_KEY"):
    os.environ["ENGRAM_API_KEY"] = getpass.getpass("ENGRAM_API_KEY: ")

# Optional: point at a non-default Engram endpoint.
# os.environ["ENGRAM_BASE_URL"] = "https://..."

The package should import and report its version:

In [ ]:
import langchain_engram

print('langchain-engram', langchain_engram.__version__)
print('exports:', langchain_engram.__all__)

### A note on scoped topics (read if writes fail)
If your Engram project's group has **scoped topics**, every write must include the
scope properties that group requires. A bare `add` then fails with:

```
APIError: group 'default': insufficient scope: missing required scope
properties [custom_scope_1 some_other] to write memories
```

The helper cell below exposes a `SCOPE_PROPERTIES` dict — fill it with the
properties your project requires and they'll be merged into every write (and used
as a search filter). Leave it `{}` if your project has no scoped topics. Each
package surface accepts the same values via its `properties=` argument.

### Round trip
Add a memory, wait for Engram's pipeline to commit it, then search it back.

In [ ]:
import time

from langchain_engram._client import build_client

client = build_client()  # reads ENGRAM_API_KEY

# If your Engram group has SCOPED TOPICS, every WRITE must include the required
# scope properties, or the API rejects it with, e.g.:
#   APIError: group 'default': insufficient scope: missing required scope
#   properties [custom_scope_1 some_other] to write memories
# Fill in the scope properties your project requires (leave {} if none):
SCOPE_PROPERTIES = {}  # e.g. {'custom_scope_1': 'demo', 'some_other': 'demo'}


def _merge_props(properties):
    merged = {**SCOPE_PROPERTIES, **(properties or {})}
    return merged or None


def seed_memory(text, user_id, *, properties=None, **kwargs):
    """Add a memory (with required scope properties) and wait for it to commit."""
    run = client.memories.add(
        text, user_id=user_id, properties=_merge_props(properties), **kwargs
    )
    status = client.runs.wait(run.run_id, timeout=60.0)
    print(f'run {run.run_id}: {status.status} '
          f'(+{len(status.memories_created)} / ~{len(status.memories_updated)})')
    return status


def wait_until_searchable(query, user_id, needle, timeout=60.0, *, properties=None, **kwargs):
    """Poll search until `needle` shows up (handles eventual consistency)."""
    props = _merge_props(properties)
    deadline = time.time() + timeout
    while time.time() < deadline:
        results = client.memories.search(
            query=query, user_id=user_id, properties=props, **kwargs
        )
        if any(needle.lower() in m.content.lower() for m in results):
            return results
        time.sleep(2.0)
    return client.memories.search(
        query=query, user_id=user_id, properties=props, **kwargs
    )

In [ ]:
USER = 'playground-user'

# seed_memory merges SCOPE_PROPERTIES into the write and waits for the pipeline.
status = seed_memory('The user is learning to play the cello.', user_id=USER)

# Did the pipeline actually store anything? (empty -> nothing was extracted/written)
print('created:', [op.memory_id for op in status.memories_created])
print('updated:', [op.memory_id for op in status.memories_updated])

Now read it back, printing each memory's `user_id` and `properties` so you can see
exactly how it was scoped. Search by the same scope properties — an unscoped search
may not surface scoped memories.

In [ ]:
results = client.memories.search(
    query='hobbies', user_id=USER, properties=SCOPE_PROPERTIES or None
)
if not results and SCOPE_PROPERTIES:
    # the topic may not be user-scoped: retry filtering by scope only
    results = client.memories.search(query='hobbies', properties=SCOPE_PROPERTIES)

print(f'{len(results)} result(s):')
for m in results:
    print(f'- ({m.score}) {m.content}')
    print(f'    user_id={m.user_id!r} | properties={m.properties}')

if not results:
    print('\nNo results. Checklist:')
    print(' - did `created`/`updated` above list a memory id? if both are empty,')
    print('   the pipeline stored nothing (e.g. nothing worth extracting).')
    print(' - does SCOPE_PROPERTIES match your topic scope exactly?')
    print(' - is the query semantically related to the memory text?')

### Why isn't it tied to a user? (topic scope)
Whether a memory is tied to a `user_id` is decided by the **topic's scope**, set when
you create the project. Engram *ignores* any scope parameter a topic doesn't declare:

- If your topic is **user-scoped**, the write attaches `user_id` and you can filter
  searches by it (`m.user_id` is set above).
- If your topic is scoped by **custom properties only** (e.g. `custom_scope_1`,
  `some_other`) and *not* by `user_id` — which is exactly the case when the write
  required `[custom_scope_1 some_other]` but **not** `user_id` — then `user_id` is
  ignored: `m.user_id` is `None`, and filtering a search by `user_id` does nothing.

To tie memories to users, **add `user_id` to the topic's scope in the Engram console**
(make the topic user-scoped). The package always sends `user_id`; Engram honors it
only when the topic is user-scoped.

### Seeing scoped data in the Engram console
Scoped memories are stored **under their scope** — the console's default/unscoped
view will not show them. Select the same scope properties (e.g. `custom_scope_1=demo`,
`some_other=demo`, plus `user_id` if the topic is user-scoped) in the console to view
them. The `created` list printed above is the source of truth that a memory was written.

If you see the cello memory above, your setup works. ✅

Move on to:
- `1_middleware_agent_memory.ipynb` — drop-in agent memory
- `2_memory_tools.ipynb` — agent-controlled memory tools
- `3_engram_store.ipynb` — the LangGraph `BaseStore` adapter
- `4_conversation_scoping.ipynb` — tie memories to a conversation
- `5_topics.ipynb` — restrict recall to topics